## ENEM: Rede Neural

-> A rede é o `MLPRegressor` da statstech: uma rede com camadas ReLU treinada por gradiente descendente em mini-lotes por um número fixo de épocas.

#### 0. Configuração inicial

In [2]:
import json
import time

import matplotlib.pyplot as plt
import numpy as np
from statstech.linear_model import Ridge
from statstech.model_selection import (
    GridSearchCV,
    KFold,
    cross_val_score,
    train_test_split,
)
from statstech.neural_network import MLPRegressor
from statstech.pipeline import Pipeline
from statstech.preprocessing import StandardScaler

from analysis.config import SEED, STYLE, TABLES
from analysis.evaluation import evaluate_regressor
from analysis.io import load_dataset
from analysis.report import save_figure, save_table

plt.style.use(STYLE)


def make_pipeline(model):
    # Padroniza X dentro de cada ajuste; o y é padronizado pela própria rede (scale_y).
    return Pipeline([("scaler", StandardScaler()), ("model", model)])


def rmse(y_true, y_pred):
    return evaluate_regressor(y_true, y_pred)["rmse"]

#### 1. Dados

In [3]:
data = load_dataset("enem")
X_train, y_train = data["X_train"], data["y_train"]
feature_names = data["feature_names"].tolist()

print(f"treino: {X_train.shape[0]} participantes, {X_train.shape[1]} colunas")
print(f"nota média: {y_train.mean():.1f} pontos, desvio padrão {y_train.std():.1f}")

treino: 38347 participantes, 41 colunas
nota média: 545.2 pontos, desvio padrão 87.7


#### 2. Validação

Temos duas formas de separar o treino: Cross Validation ou divisão fixa do terino em 80% e 20% por exemplo. 
O cv (cross validation) escolhe e compara já a divisão fixa auxiliaria em diagnósticos rápidos. 

In [4]:
cv = KFold(5, shuffle=True, random_state=SEED)
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.2,
                                              random_state=SEED)
print(f"diagnóstico: {len(y_fit)} para ajustar, {len(y_val)} para validar")

diagnóstico: 30677 para ajustar, 7670 para validar


#### 3. Baselines: Média e Ridge

Para podermos comparar os resultados da rede neural com outros modelos, por exemplo, a média e o Ridge (melhor modelo linear)

In [6]:
mean_scores = []
for fit_idx, val_idx in cv.split(X_train):
    prediction = np.full(len(val_idx), y_train[fit_idx].mean())
    mean_scores.append(rmse(y_train[val_idx], prediction))
mean_scores = np.array(mean_scores)

ridge_search = GridSearchCV(make_pipeline(Ridge()),
                            {"model__alpha": list(np.logspace(-2, 4, 13))},
                            cv=cv, scoring="rmse").fit(X_train, y_train)
ridge_best = ridge_search.cv_results_[ridge_search.best_index_]
#TODO conferir com o resultado da regularização usando o Ridge

print(f"média do treino   RMSE {mean_scores.mean():.2f} ± {mean_scores.std(ddof=1):.2f}")
print(f"Ridge (alpha={ridge_search.best_params_['model__alpha']:.3g})  "
      f"RMSE {ridge_best['mean_score']:.2f} ± {ridge_best['std_score']:.2f}")

média do treino   RMSE 87.74 ± 0.61
Ridge (alpha=316)  RMSE 72.87 ± 0.83
